In [1]:
import pandas as pd
import os
import re

In [ ]:
# def combine_snapshots(pokemon_name, folder="../pokemon_snapshots"):
#     dfs = []

#     # Normalize name for matching
#     safe_name = pokemon_name.replace(" ", "_").lower()

#     for filename in os.listdir(folder):
#         if filename.endswith(".csv") and filename.startswith(safe_name):
#             filepath = os.path.join(folder, filename)
#             df = pd.read_csv(filepath)

#             # Extract date from filename: psyduck_pricing_2026-07-10.csv
#             match = re.search(r"(\d{4}-\d{2}-\d{2})", filename)
#             snapshot_date = match.group(1) if match else None
#             df["snapshot_date"] = snapshot_date

#             dfs.append(df)

#     if not dfs:
#         raise ValueError(f"No CSV snapshots found for Pokémon: {pokemon_name}")

#     combined = pd.concat(dfs, ignore_index=True)

#     # Sort by card + date
#     combined = combined.sort_values(["id", "snapshot_date"]).reset_index(drop=True)

#     # Remove exact duplicates
#     combined = combined.drop_duplicates()

#     return combined

In [2]:

def combine_snapshots(name, folder="../pokemon_snapshots"):
    dfs = []

    # Normalize name for matching
    safe_name = name.replace(" ", "_").lower()

    for filename in os.listdir(folder):
        if filename.endswith(".csv") and filename.startswith(safe_name):
            filepath = os.path.join(folder, filename)
            df = pd.read_csv(filepath)

            dfs.append(df)

    if not dfs:
        raise ValueError(f"No CSV snapshots found for: {name}")

    # Concatenate all snapshots
    combined = pd.concat(dfs, ignore_index=True)

    # Sort chronologically using embedded timestamps
    if "cm_updated" in combined.columns:
        combined["sort_timestamp"] = combined["cm_updated"].fillna(combined.get("tp_updated"))
    else:
        combined["sort_timestamp"] = combined.get("tp_updated")

    combined = combined.sort_values(["id", "sort_timestamp"]).reset_index(drop=True)

    # Remove exact duplicates (if any should exist, which could happen) 
    combined = combined.drop_duplicates()

    # Drop helper column
    combined = combined.drop(columns=["sort_timestamp"])

    return combined


In [ ]:
## Option for excluding specific sets ##

# EXCLUDED_SETS = {
#     "Promos-A",
#     "Genetic Apex",
#     "Mythical Island",
#     "Space-Time Smackdown",
#     "Triumphant Light",
#     "Shining Revelry",
#     "Celestial Guardians",
#     "Extradimensional Crisis",
#     "Eevee Grove",
#     "Wisdom of Sea and Sky",
#     "Secluded Springs",
#     "Deluxe Pack: ex",

#     "Promos-B",
#     "Mega Rising",
#     "Crimson Blaze",
#     "Fantastical Parade",
#     "Paldean Wonders",
#     "Mega Shine",
#     "Pulsing Aura",
#     "Paradox Drive",
#     "Everyday Wonders",
#     "Ruler of the Skies",
# }

# def remove_tcg_pocket_cards(df):
#     if "set" not in df.columns:
#         return df  # nothing to filter

#     return df[~df["set"].isin(EXCLUDED_SETS)].reset_index(drop=True)

In [3]:
PRICING_COLS = [
    # Cardmarket
    "cm_avg", "cm_low", "cm_trend", "cm_avg1", "cm_avg7", "cm_avg30",
    "cm_avg_holo", "cm_low_holo", "cm_trend_holo",
    "cm_avg1_holo", "cm_avg7_holo", "cm_avg30_holo",

    # TCGPlayer - holofoil
    "tp_holofoil_lowPrice", "tp_holofoil_midPrice", "tp_holofoil_highPrice",
    "tp_holofoil_marketPrice", "tp_holofoil_directLowPrice",

    # TCGPlayer - normal
    "tp_normal_lowPrice", "tp_normal_midPrice", "tp_normal_highPrice",
    "tp_normal_marketPrice", "tp_normal_directLowPrice",

    # TCGPlayer - reverse holofoil
    "tp_reverse_holofoil_lowPrice", "tp_reverse_holofoil_midPrice",
    "tp_reverse_holofoil_highPrice", "tp_reverse_holofoil_marketPrice",
    "tp_reverse_holofoil_directLowPrice",

    # TCGPlayer - 1st edition
    "tp_1st_edition_lowPrice", "tp_1st_edition_midPrice",
    "tp_1st_edition_highPrice", "tp_1st_edition_marketPrice",
    "tp_1st_edition_directLowPrice",

    # TCGPlayer - unlimited
    "tp_unlimited_lowPrice", "tp_unlimited_midPrice",
    "tp_unlimited_highPrice", "tp_unlimited_marketPrice",
    "tp_unlimited_directLowPrice",
]


In [5]:

def preprocess_combined_df(df, name, output_folder="../Data"):
    # Removing columns that are entirely NaN
    df = df.dropna(axis=1, how="all")

    # Removing columns that are entirely empty strings
    empty_cols = [
        col for col in df.columns
        if df[col].astype(str).str.strip().eq("").all()
    ]
    df = df.drop(columns=empty_cols)

    # Removing rows with no pricing at all
    #pricing_cols = [col for col in df.columns if col.startswith(("cm_", "tp_"))]
    pricing_cols = [col for col in PRICING_COLS if col in df.columns]
    df = df.dropna(subset=pricing_cols, how="all").reset_index(drop=True)

    # --- SAVE CLEANED DATAFRAME ---
    os.makedirs(output_folder, exist_ok=True)

    safe_name = name.replace(" ", "_").lower()
    output_path = os.path.join(output_folder, f"{safe_name}_combined_clean.csv")

    df.to_csv(output_path, index=False)

    return df, output_path


In [6]:
# Loop through Pokemon snapshots
pokemon_list = ["psyduck", "pikachu", "kadabra"]

for pokemon in pokemon_list:
    combined_df = combine_snapshots(pokemon)
    clean_df, output_path = preprocess_combined_df(combined_df, pokemon)
    print(f"Saved cleaned combined file for {pokemon} to: {output_path}\n")


Saved cleaned combined file for psyduck to: ../Data/psyduck_combined_clean.csv

Saved cleaned combined file for pikachu to: ../Data/pikachu_combined_clean.csv

Saved cleaned combined file for kadabra to: ../Data/kadabra_combined_clean.csv



In [7]:
# loop through set snapshots
set_list = ["chaos_rising", "pitch_black"]

for set_name in set_list:
    combined_set = combine_snapshots(set_name, folder="../set_snapshots")
    clean_set, output_path = preprocess_combined_df(combined_set, set_name)
    print(f"Saved cleaned combined file for {set_name} to: {output_path}\n")


Saved cleaned combined file for chaos_rising to: ../Data/chaos_rising_combined_clean.csv

Saved cleaned combined file for pitch_black to: ../Data/pitch_black_combined_clean.csv



In [ ]:
# Filling out missing rarities 

## Pikachu (19 cards) ##
pikachu_df = pd.read_csv('../Data/pikachu_combined_clean.csv')

rarity_updates = {
    "tk-ex-latio-6": "Common",
    "2014xy-5": "Promo",
    "2015xy-6": "Promo",
    "2016xy-6": "Promo",
    "2017sm-5": "Promo",
    "2018sm-4": "Promo",
    "2019sm-6": "Promo",
    "2021swsh-25": "Promo",
    "2022swsh-7": "Promo",
    "2023sv-6": "Promo",
    "2024sv-2": "Promo",
    "ex5.5-5": "Promo",
    "fut2020-1": "Promo",
    "mfb-17": "Promo",
    "ru1-7": "Promo",
    "tk-sm-r-14": "Promo",
    "tk-sm-r-29": "Promo",
    "tk-xy-p-14": "Promo",
    "tk-xy-p-30": "Promo",
}

for card_id, rarity in rarity_updates.items():
    pikachu_df.loc[pikachu_df["id"] == card_id, "rarity"] = rarity


pikachu_df.to_csv('../Data/pikachu2_combined_clean.csv', index=False)


## Psyduck (1 card) ##
psyduck_df = pd.read_csv('../Data/psyduck_combined_clean.csv')
psyduck_df.loc[psyduck_df["id"] == "2018sm-2", "rarity"] = "Promo"
psyduck_df.to_csv('../Data/psyduck2_combined_clean.csv', index=False)


